In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("RDD_Lab") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext

In [ ]:
file_path = "file:///home/itversity/employees.txt"

In [ ]:
def clean_line(line):
    """تنظيف الأسطر وإصلاح الأخطاء"""
    if not line or line.strip() == "":
        return None
    # إصلاح أي سطر به مشكلة
    if "Legal Counsel145000" in line:
        line = line.replace("Legal Counsel145000", "Legal Counsel,145000")
    parts = line.split(",")
    if len(parts) >= 9:
        return line
    return None

In [ ]:
cleaned_rdd = text_rdd.map(clean_line).filter(lambda x: x is not None)

In [ ]:
# إزالة الرأس (السطر الأول)
header = cleaned_rdd.first()
data_rdd = cleaned_rdd.filter(lambda x: x != header)

In [ ]:
print(f" عدد السجلات بعد التنظيف: {data_rdd.count()}")

In [ ]:
# ============================================
# التمرين 1: تحويل البيانات إلى بنية منظمة
# ============================================
print("\n" + "="*60)
print("【التمرين 1】تحويل البيانات (map + split)")
print("="*60)

parsed_rdd = data_rdd.map(lambda x: x.split(","))

print("أول 5 سجلات بعد التحويل:")
for i, record in enumerate(parsed_rdd.take(5)):
    print(f"   {i+1}. ID: {record[0]}, Name: {record[1]}, Dept: {record[2]}, Salary: ${record[4]}")

In [ ]:
# ============================================
# التمرين 2: حساب عدد مرات ظهور كل اسم
# ============================================
print("\n" + "="*60)
print("【التمرين 2】عدد مرات ظهور كل اسم (map + reduceByKey)")
print("="*60)

name_counts = data_rdd.map(lambda x: (x.split(",")[1], 1)) \
                      .reduceByKey(lambda a, b: a + b)

print("عدد مرات ظهور كل اسم:")
for name, count in name_counts.collect():
    print(f"   {name}: {count} مرة")

In [ ]:
# ============================================
# التمرين 3: تصفية السجلات غير الصالحة
# ============================================
print("\n" + "="*60)
print("【التمرين 3】تصفية السجلات (filter)")
print("="*60)

# تصفية السجلات الصالحة (الراتب > 0)
valid_rdd = data_rdd.filter(lambda x: float(x.split(",")[4]) > 0)

print(f" السجلات الصالحة: {valid_rdd.count()}")
print(f" السجلات غير الصالحة: {data_rdd.count() - valid_rdd.count()}")

print("\nأول 5 سجلات صالحة:")
for line in valid_rdd.take(5):
    parts = line.split(",")
    print(f"   {parts[0]}. {parts[1]} - {parts[2]} - ${parts[4]}")

In [ ]:
# ============================================
# التمرين 4: حساب متوسط الراتب لكل قسم
# ============================================
print("\n" + "="*60)
print("【التمرين 4】متوسط الراتب لكل قسم (map + reduceByKey)")
print("="*60)

# استخراج القسم والراتب
dept_salary = valid_rdd.map(lambda x: (x.split(",")[2], float(x.split(",")[4])))

# حساب المجموع والعدد
dept_sum_count = dept_salary.mapValues(lambda x: (x, 1)) \
                             .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# حساب المتوسط
dept_avg = dept_sum_count.mapValues(lambda x: x[0] / x[1]) \
                         .sortBy(lambda x: x[1], ascending=False)

print("متوسط الراتب لكل قسم:")
for dept, avg in dept_avg.collect():
    print(f"   {dept}: ${avg:,.2f}")

In [ ]:
# ============================================
# التمرين 5: حساب عدد الموظفين لكل قسم
# ============================================
print("\n" + "="*60)
print("【التمرين 5】عدد الموظفين لكل قسم (map + reduceByKey)")
print("="*60)

dept_count = valid_rdd.map(lambda x: (x.split(",")[2], 1)) \
                      .reduceByKey(lambda a, b: a + b) \
                      .sortBy(lambda x: x[1], ascending=False)

print("عدد الموظفين لكل قسم:")
total = valid_rdd.count()
for dept, count in dept_count.collect():
    percentage = (count / total) * 100
    print(f"   {dept}: {count} موظف ({percentage:.1f}%)")

In [ ]:
# ============================================
# عرض ملخص النتائج
# ============================================
print("\n" + "="*60)
print(" ملخص النتائج النهائي")
print("="*60)

print(f"""
 إجمالي السجلات في الملف: {data_rdd.count()}
 السجلات الصالحة: {valid_rdd.count()}
 السجلات غير الصالحة: {data_rdd.count() - valid_rdd.count()}

 أعلى قسم من حيث عدد الموظفين: {dept_count.first()[0]} ({dept_count.first()[1]} موظف)
 أعلى قسم من حيث متوسط الراتب: {dept_avg.first()[0]} (${dept_avg.first()[1]:,.2f})
""")